# 训练轮次收益差的分层诊断
复用已完成Qlib模型输出和原生账本，不重训、不新增策略回测。原始/平滑输出局部得分与账户状态分组分别分析，不作因果归因。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.minute_pit_allocator import _ensemble
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/validation_epoch_path_diagnostic_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
x,xa=_source(root,p['source'],['analysis/predictions','analysis/targets',*[f'analysis/native/{family}_{cost}' for family in ['candidate','control'] for cost in p['slippage_bps']]]);prices,pa=_source(root,p['price_source'],['inputs/frame']);frame=prices['inputs/frame'].set_index('signal_date');reference=x['analysis/native/control_2']['authority'];dates=pd.DatetimeIndex(reference.signal_date);frame=frame.reindex(dates);assert frame[['soxl_open','soxx_open','soxl_exit_open','soxx_exit_open']].notna().all().all();returns=pd.DataFrame({'soxl_weight':frame.soxl_exit_open/frame.soxl_open-1,'soxx_weight':frame.soxx_exit_open/frame.soxx_open-1},index=dates)
raw={f:_ensemble(x['analysis/predictions'],f,p['seeds']).reindex(dates) for f in ['control','candidate']};targets={f:x['analysis/targets'][f].reindex(dates) for f in ['control','candidate']};fields=list(raw['control']);changes=[];local=pd.DataFrame(index=dates)
for stage,values in [('raw',raw),('smoothed',targets)]:
 for field in fields:
  diff=values['candidate'][field]-values['control'][field];assert diff.notna().all();changes.append({'stage':stage,'field':field,'mean_signed_difference_pp':100*diff.mean(),'mean_absolute_difference_pp':100*diff.abs().mean(),'maximum_absolute_difference_pp':100*diff.abs().max()})
 delta=values['candidate'][list(returns)]-values['control'][list(returns)];local[stage+'_gross_local_advantage']=(delta*returns).sum(axis=1)
assert np.isfinite(local).all().all()
# Bootstrap scalar local diagnostics on the full contiguous paired calendar; no account reinterpretation.
rng=np.random.default_rng(p['bootstrap']['seed']);values=local.to_numpy();n=len(local);k=p['bootstrap']['resamples'];idx=rng.integers(n,size=k);totals=values[idx].copy()
for _ in range(1,n):
 restart=rng.random(k)<1/p['bootstrap']['mean_block'];idx=(idx+1)%n;idx[restart]=rng.integers(n,size=int(restart.sum()));totals+=values[idx]
means=totals/n;local_rows=[]
for j,name in enumerate(local):
 lo,hi=np.quantile(means[:,j],[p['bootstrap']['alpha'],1-p['bootstrap']['alpha']]);local_rows.append({'score':name,'sessions':n,'daily_mean_bp':1e4*values[:,j].mean(),'daily_lower_bp':1e4*lo,'daily_upper_bp':1e4*hi,'lower_positive':bool(lo>0),'is_net_strategy_return':False})
partitions=[];divergences=[];daily={}
for cost in p['slippage_bps']:
 a=x[f'analysis/native/candidate_{cost}']['authority'];b=x[f'analysis/native/control_{cost}']['authority'];assert a.signal_date.equals(b.signal_date) and a.trade_date.equals(b.trade_date);d=pd.DataFrame({'signal_date':a.signal_date,'trade_date':a.trade_date,'log_difference':np.log1p(a.strategy_return)-np.log1p(b.strategy_return),'risk_state_differs':a.risk_state.ne(b.risk_state),'risk_clock_differs':a.risk_clock.ne(b.risk_clock),'open_action_differs':a.execute_open_target.ne(b.execute_open_target),'recovery_action_differs':a.recovery_action.ne(b.recovery_action),'cap_differs':~np.isclose(a.current_beta_cap,b.current_beta_cap,equal_nan=True)});daily[cost]=d
 for field in ['risk_state_differs','risk_clock_differs','open_action_differs','recovery_action_differs','cap_differs']:
  m=d[field];divergences.append({'slippage_bps':cost,'field':field,'sessions':int(m.sum()),'first_trade':str(d.loc[m,'trade_date'].min().date()) if m.any() else None})
 for year in ['all',2023,2024,2025,2026]:
  mask=np.ones(len(d),dtype=bool) if year=='all' else d.trade_date.dt.year.eq(year).to_numpy();total=d.loc[mask,'log_difference'].sum();subtotal=0.
  for differs in [False,True]:
   m=mask & d.risk_state_differs.eq(differs).to_numpy();value=float(d.loc[m,'log_difference'].sum());subtotal+=value;partitions.append({'slippage_bps':cost,'trade_year':year,'risk_state_differs':differs,'sessions':int(m.sum()),'log_difference_sum':value,'all_state_log_difference_sum':float(total),'descriptive_not_causal':True})
  assert np.isclose(total,subtotal,rtol=0,atol=1e-12)
 for asset in ['soxl','soxx','cash']:
  diff=a['ending_'+asset+'_weight']-b['ending_'+asset+'_weight'];changes.append({'stage':f'actual_ending_{cost}bp','field':asset+'_weight','mean_signed_difference_pp':100*diff.mean(),'mean_absolute_difference_pp':100*diff.abs().mean(),'maximum_absolute_difference_pp':100*diff.abs().max()})
changes=pd.DataFrame(changes);local_table=pd.DataFrame(local_rows);partition=pd.DataFrame(partitions);divergence=pd.DataFrame(divergences);policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily_paths','analysis/local_scores']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,source_audits={'parent':xa,'prices':pa});rec.save_objects(artifact_path='analysis',daily_paths=daily,local_scores=local,changes=changes,local_summary=local_table,state_partition=partition,divergence=divergence);rec.log_metrics(new_fits=0,new_native_cases=0,registered_artifact_count=9)
summary={'experiment':card.name,'decision':'diagnostic_only_parent_remains_not_promoted','new_fits':0,'new_native_cases':0,'raw_local_advantage_lower_positive':bool(local_table.iloc[0].lower_positive),'objective_achieved':False,'provider_uri':str(provider),'region':'us','as_of':p['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':p['mechanism']}
rec.save_objects(artifact_path='analysis',summary=summary)
rec.log_metrics(registered_artifact_count=sum(f.is_file() for f in (Path(summary['recorder_local_dir'])/'artifacts').rglob('*')))
for name,value in [('prediction_changes',changes),('local_score_summary',local_table),('state_partition',partition),('state_divergence',divergence)]:value.to_csv(card/(name+'.csv'),index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}));a=Path(summary['recorder_local_dir'])/'artifacts';atomic_json({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');print(changes.to_string(index=False));print(local_table.to_string(index=False));print(partition[partition.trade_year.eq('all')].to_string(index=False));print(divergence.to_string(index=False))


# 训练轮次改进的路径诊断：局部得分与实际仓位均改变

复用上一轮6套完整Qlib原生账户及3seed预测，不新增拟合或交易回测，不改变任何参数。分析覆盖原944区间、2023-01-03信号至2026-10-09退出，IBKR Tiered/10万美元及三个独立成本路径均保持原身份。本卡是解释已完成训练回测，不是新策略，也不是新的独立证据。

原始候选相对末轮对照：现金目标平均低1.10个百分点（平均绝对差1.41、最大绝对差12.02）；SOXL平均高0.37、SOXX高0.74个百分点。平滑后现金平均低1.14个百分点。

按同日09:47至下一交易日09:47收益计算的ΣΔ权重×资产收益，原始输出每日平均+0.836bp，63日均长/2000次区块区间[+0.372,+1.365]bp；平滑后+0.496bp，区间[+0.042,+1.026]bp。这是未扣费、没有持仓漂移或风控约束的局部配置得分，不是回测净收益，不做复利账户。它包含平均风险暴露增加，未分离择时与暴露补偿，也未计模型选择/重训不确定性。因此只能说既有路径上的局部得分为正，不能称独立预测alpha。

2bp实际期末现金平均低8.68个百分点，平均绝对差10.31、最大绝对差60.41；明显大于原始目标差异。321/944日两账户risk_state不同，首次2024-07-24；risk_clock不同373日，开盘execute/HOLD不同130日，恢复动作不同0日。0bp状态差321日，5bp差349日，后者首次2023-04-26。beta_cap连续值更早就有差异（2bp768日）。

2bp完整账户累计对数收益差0.674793，其中状态相同日期0.195818、状态不同日期0.478975，后者约71%。三成本及全部交易年度的分组总和与原账本差值逐项对上。状态相同并不代表实际仓位、cap、历史峰值或路由相同；状态不同分组是内生的事后分组。该71%不是“风控造成71%提升”的因果分解，不能据此删风控或调触发线。

结论：训练轮次选择同时改变局部配置得分和完整策略状态路径；微小验证损失差可对应大幅实际持仓差。既不能把改善全说成控制器偶然性，也不能把全部账户超额说成更好的预测。父候选仍落后强参考及SOXX，保持not_promoted；本诊断不晋升。下一项实质需求是训练期时间外的风险状态条件预测与成本后价值证据，不能凭本次分组直接修改阈值，也不继续选择同一历史上的epoch。

完整逐日诊断存Qlib Recorder/NAS；小表在根目录，中文结论与notebook一致。NAS冷恢复逐对象哈希通过，原账户日期一致、价格完整有限及状态分组恒等式已在notebook执行中检查。本轮没有新增业务合同，不追加软件测试来替代研究证据。
